# ZenFit Meal Classifier — Colab GPU Workflow
Run cells in order. Full training and uploads are opt-in. This notebook never changes `active.json` or production settings.

## 1. Runtime verification

In [1]:
import os, sys, json, platform, subprocess, hashlib, shutil, time
from pathlib import Path
print({'python': sys.version, 'platform': platform.platform(), 'cwd': os.getcwd()})
IN_COLAB = 'google.colab' in sys.modules
print('Google Colab runtime:', IN_COLAB)

{'python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35', 'cwd': '/content'}
Google Colab runtime: True


## 2. Repository setup

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_PATH = Path("/content/ZenFit")

# Put your PUBLIC GitHub repo URL here.
# Do not include tokens in the URL.
REPO_URL = "https://github.com/YOUR_USERNAME/ZenFit.git"

if not REPO_PATH.exists():
    print("Cloning ZenFit repository into Colab runtime...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_PATH)],
        check=True
    )
else:
    print("Repository already exists:", REPO_PATH)

BACKEND_PATH = REPO_PATH / "backend"

if not (BACKEND_PATH / "training").is_dir():
    raise FileNotFoundError(
        f"Could not find training directory at "
        f"{BACKEND_PATH / 'training'}"
    )

os.chdir(BACKEND_PATH)

if str(BACKEND_PATH) not in sys.path:
    sys.path.insert(0, str(BACKEND_PATH))

print("Repository:", REPO_PATH)
print("Backend:", BACKEND_PATH)
print("Current directory:", Path.cwd())

FileNotFoundError: Set ZENFIT_REPO_PATH to an existing checkout or ZENFIT_REPO_URL to clone

## 3. Dependency installation

In [ ]:
INSTALL_DEPS = False  # set True once per fresh runtime
if INSTALL_DEPS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-training.txt'], check=True)
    # Colab already provides CUDA-compatible torch. Only install requirements-ai.txt if imports below prove it is needed.
import torch, torchvision, numpy, pandas, sklearn, PIL
print({'torch':torch.__version__, 'torchvision':torchvision.__version__, 'numpy':numpy.__version__, 'pandas':pandas.__version__, 'sklearn':sklearn.__version__, 'Pillow':PIL.__version__})

## 4. GPU verification

In [ ]:
available = torch.cuda.is_available()
print('torch.cuda.is_available():', available)
print('CUDA version:', torch.version.cuda)
if available:
    props=torch.cuda.get_device_properties(0)
    print('GPU name:', torch.cuda.get_device_name(0))
    print('Total GPU memory (GiB):', round(props.total_memory/2**30,2))
    print('Allocated memory (GiB):', round(torch.cuda.memory_allocated()/2**30,3))
    print('Reserved memory (GiB):', round(torch.cuda.memory_reserved()/2**30,3))
else: raise RuntimeError('CUDA unavailable. Select a Colab GPU runtime before any training cell.')

## 5. Dataset configuration

In [ ]:
LOCAL_ROOT=Path(os.getenv('ZENFIT_COLAB_ROOT','/content/zenfit-work'))
RAW_ROOT=LOCAL_ROOT/'data/raw/kaggle'; DATASET=LOCAL_ROOT/'data/training/indian_food_v2'; REPORTS=LOCAL_ROOT/'reports'
MODELS=LOCAL_ROOT/'models/indian_food'; PACKAGES=LOCAL_ROOT/'artifacts'; DRIVE_ROOT=None
for p in (RAW_ROOT,REPORTS,MODELS,PACKAGES): p.mkdir(parents=True,exist_ok=True)
PRIMARY='harishkumardatalab/food-image-classification-dataset'
print({'primary':PRIMARY,'dataset':str(DATASET),'raw_images_are_gitignored':True})

## 6. Dataset acquisition

In [ ]:
DOWNLOAD_DATASET=False
# Authentication options: KAGGLE_API_TOKEN env var, a secure ~/.kaggle/kaggle.json, or a Colab secret copied without printing.
if DOWNLOAD_DATASET:
    if not (os.getenv('KAGGLE_API_TOKEN') or (Path.home()/'.kaggle/kaggle.json').exists()): raise RuntimeError('Configure Kaggle credentials securely')
    subprocess.run([sys.executable,'training/download_kaggle_datasets.py','--dataset','food_image_classification','--root',str(RAW_ROOT)],check=True)

## 7. Dataset audit

In [ ]:
PREPARE_DATASET=False
raw=RAW_ROOT/'food_image_classification/Food Classification dataset'
if PREPARE_DATASET:
    subprocess.run([sys.executable,'training/prepare_class_labeled_v2.py','--raw',str(raw),'--output',str(DATASET),'--reports',str(REPORTS)],check=True)
manifest=json.loads((DATASET/'split_manifest.json').read_text()) if (DATASET/'split_manifest.json').exists() else None
print('Manifest ready:', manifest is not None)

## 8. Train/validation/test split verification

In [ ]:
from collections import Counter
if manifest is None: raise FileNotFoundError('Acquire and prepare the dataset first')
split_counts=Counter(Path(x['path']).parts[0] for x in manifest['files'])
assert set(split_counts)=={'train','val','test'}
print(dict(split_counts), manifest['ratios'])

## 9. Class distribution

In [ ]:
class_distribution={name:{k:v for k,v in row.items() if k in ('total','train','val','test')} for name,row in manifest['classes'].items()}
print(json.dumps(class_distribution,indent=2))

## 10. Duplicate/leakage checks

In [ ]:
hash_splits={}
for row in manifest['files']:
    split=Path(row['path']).parts[0]; digest=row['sha256']; hash_splits.setdefault(digest,set()).add(split)
leaks={h:sorted(s) for h,s in hash_splits.items() if len(s)>1}
assert not leaks, f'Split leakage detected: {len(leaks)} hashes'
assert len(hash_splits)==len(manifest['files']), 'Duplicate content remains in the prepared set'
print('No SHA256 duplicates or split leakage across',len(hash_splits),'images')

## 11. Model configuration

In [ ]:
CONFIG=Path('training/configs/indian_food_v2_candidate.json')
cfg=json.loads(CONFIG.read_text()); VERSION=os.getenv('ZENFIT_MODEL_VERSION','1.2.0-colab-candidate')
print(json.dumps(cfg,indent=2)); print('Candidate output:',MODELS/VERSION)

## 12. Baseline model loading

In [ ]:
BASELINE_VERSION=os.getenv('ZENFIT_BASELINE_VERSION','1.1.0')
baseline=MODELS/BASELINE_VERSION
print('Compatible baseline checkpoint available:',(baseline/'model.pt').exists())
# The canonical trainer currently uses ImageNet EfficientNet-B0 initialization. Existing candidates remain comparison baselines unless an explicit resume path is added.

## 13. Training

In [ ]:
RUN_GPU_SMOKE=False; RUN_FULL_TRAINING=False
def train(version, smoke):
    cmd=[sys.executable,'training/train_indian_food.py',str(DATASET),'--models-dir',str(MODELS),'--config',str(CONFIG),'--version',version,'--dataset-version',manifest['dataset_version'],'--device','cuda','--require-cuda','--num-workers','2']
    if smoke: cmd += ['--smoke','--smoke-samples-per-split','128']
    subprocess.run(cmd,check=True); return MODELS/version
if RUN_GPU_SMOKE: SMOKE_ROOT=train(VERSION+'-smoke',True)
if RUN_FULL_TRAINING:
    if not RUN_GPU_SMOKE and not (MODELS/(VERSION+'-smoke')/'metrics.json').exists(): raise RuntimeError('Complete GPU smoke training first')
    CANDIDATE=train(VERSION,False)
else: CANDIDATE=MODELS/VERSION
print('No training launched automatically. Candidate:',CANDIDATE)

## 14. Validation

In [ ]:
metrics=json.loads((CANDIDATE/'metrics.json').read_text()) if (CANDIDATE/'metrics.json').exists() else {}
history=metrics.get('history',[]); print('Best epoch:',metrics.get('best_epoch'),'best validation accuracy:',metrics.get('best_validation_accuracy'))

## 15. Test evaluation

In [ ]:
print({k:metrics.get(k) for k in ('sample_count','accuracy','balanced_accuracy','macro_precision','macro_recall','macro_f1','top_3_accuracy')})

## 16. Confusion matrix

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
if metrics:
    classes=json.loads((CANDIDATE/'classes.json').read_text()); matrix=json.loads((CANDIDATE/'confusion_matrix.json').read_text())
    plt.figure(figsize=(12,10)); sns.heatmap(matrix,cmap='Blues',xticklabels=classes,yticklabels=classes); plt.tight_layout(); plt.savefig(REPORTS/f'{VERSION}-confusion-matrix.png',dpi=160); plt.show()

## 17. Per-class metrics

In [ ]:
if metrics: display(pandas.DataFrame(metrics['per_class']).T.sort_values('f1-score'))

## 18. Top-1 accuracy

In [ ]:
print('Top-1 accuracy:',metrics.get('accuracy'))

## 19. Top-3 accuracy

In [ ]:
print('Top-3 accuracy:',metrics.get('top_3_accuracy'))

## 20. Macro F1

In [ ]:
print('Macro F1:',metrics.get('macro_f1'))

## 21. Calibration

In [ ]:
cal=json.loads((CANDIDATE/'calibration.json').read_text()) if (CANDIDATE/'calibration.json').exists() else {}
print({k:cal.get(k) for k in ('temperature','ece','brier_score')})

## 22. Open-set evaluation

In [ ]:
OPEN_SET_EVIDENCE=REPORTS/f'{VERSION}-open-set-predictions.json'
# Evidence schema uses truth=supported_food|unknown_food|non_food plus top_candidates, entropy, optional food_probability/energy_score.
print('Evidence exists:',OPEN_SET_EVIDENCE.exists())

## 23. Non-food evaluation

In [ ]:
if OPEN_SET_EVIDENCE.exists():
    rows=json.loads(OPEN_SET_EVIDENCE.read_text()).get('predictions',[])
    print('Non-food samples:',sum(r.get('truth')=='non_food' for r in rows))
else: print('Add reviewed non-food evidence; synthetic probes alone cannot satisfy production gates.')

## 24. Threshold search

In [ ]:
if OPEN_SET_EVIDENCE.exists():
    from training.open_set_evaluation import threshold_sweep
    from training.analyze_open_set_thresholds import recommend
    recommendation=recommend(threshold_sweep(rows,VERSION,confidence_values=(.5,.57,.6,.65,.7),margin_values=(.05,.1,.15,.2),entropy_values=(None,1.0,1.5,2.0)))
    recommendation['status']='DEVELOPER_BETA_CANDIDATE'; (REPORTS/f'{VERSION}-threshold-report.json').write_text(json.dumps(recommendation,indent=2)); print(json.dumps(recommendation,indent=2))

## 25. Candidate comparison

In [ ]:
subprocess.run([sys.executable,'training/compare_indian_food_models.py',BASELINE_VERSION,VERSION,'--models-dir',str(MODELS)],check=False)

## 26. Artifact export

In [ ]:
EXPORT_ARTIFACT=False
if EXPORT_ARTIFACT:
    threshold_report=json.loads((REPORTS/f'{VERSION}-threshold-report.json').read_text()); (CANDIDATE/'open_set_thresholds.json').write_text(json.dumps(threshold_report['thresholds'],indent=2))
    destination=PACKAGES/VERSION; subprocess.run([sys.executable,'scripts/package_model_artifact.py',str(CANDIDATE),str(destination),'--environment','developer-beta'],check=True)
    from app.ai.artifacts import verify_artifact; artifact_manifest=verify_artifact(destination,required_environment='developer-beta'); print(destination,artifact_manifest)

## 27. Model card generation

In [ ]:
if (CANDIDATE/'model_card.md').exists(): print((CANDIDATE/'model_card.md').read_text())

## 28. Promotion readiness report

In [ ]:
if CANDIDATE.exists(): subprocess.run([sys.executable,'training/promote_indian_food.py',VERSION,'--environment','developer-beta','--models-dir',str(MODELS)],check=False)
print('Production remains blocked unless every production gate passes. active.json is never written by this cell.')

## 29. Optional artifact upload

In [ ]:
MOUNT_DRIVE=False; COPY_TO_DRIVE=False
if MOUNT_DRIVE:
    from google.colab import drive; drive.mount('/content/drive'); DRIVE_ROOT=Path('/content/drive/MyDrive/ZenFit')
if COPY_TO_DRIVE:
    if DRIVE_ROOT is None: raise RuntimeError('Mount Drive first')
    target=DRIVE_ROOT/'artifacts'/VERSION
    if target.exists(): raise FileExistsError(f'Refusing to overwrite {target}')
    shutil.copytree(PACKAGES/VERSION,target); print('Copied to',target)
# For configured S3-compatible storage, use app.ai.artifacts.S3CompatibleArtifactStorage; credentials stay in environment/secrets.

## 30. Cleanup

In [ ]:
RUN_INFERENCE_SMOKE=False
if RUN_INFERENCE_SMOKE:
    from app.ai.artifacts import verify_artifact
    from app.ai.meal_scan.open_set import Candidate,OpenSetDecisionEngine,OpenSetInput,OpenSetThresholds,probability_entropy
    from torchvision.models import efficientnet_b0,EfficientNet_B0_Weights
    bundle=PACKAGES/VERSION; verify_artifact(bundle,required_environment='developer-beta')
    labels=json.loads((bundle/'classes.json').read_text()); temperature=json.loads((bundle/'calibration.json').read_text())['temperature']
    thresholds=OpenSetThresholds.from_json(bundle/'open_set_thresholds.json'); exported=efficientnet_b0(num_classes=len(labels)).cuda()
    exported.load_state_dict(torch.load(bundle/'model.pt',map_location='cuda',weights_only=True)); exported.eval(); transform=EfficientNet_B0_Weights.DEFAULT.transforms()
    sample_groups={'known':list((DATASET/'test').glob('*/*'))[:5],'unknown':list((raw/'burger').glob('*'))[:3]+list((raw/'pizza').glob('*'))[:3],'non_food':list((LOCAL_ROOT/'open_set/non_food').glob('*'))[:6]}
    smoke_rows=[]
    for group,paths in sample_groups.items():
        for path in paths:
            image=PIL.Image.open(path).convert('RGB'); torch.cuda.synchronize(); start=time.perf_counter()
            with torch.inference_mode(): probs=(exported(transform(image).unsqueeze(0).cuda())/temperature).softmax(1)[0].cpu()
            torch.cuda.synchronize(); values,indices=probs.topk(min(3,len(labels))); candidates=tuple(Candidate(labels[int(i)],float(v)) for v,i in zip(values,indices))
            decision=OpenSetDecisionEngine(thresholds).decide(OpenSetInput(top_candidates=candidates,entropy=probability_entropy(probs),model_version=VERSION))
            row={'group':group,'file':path.name,'predicted_class':candidates[0].label,'confidence':candidates[0].confidence,'top_3':[{'label':x.label,'confidence':x.confidence} for x in candidates],'decision':decision.decision.value,'latency_ms':(time.perf_counter()-start)*1000}; smoke_rows.append(row); print(row)
    (REPORTS/f'{VERSION}-independent-inference-smoke.json').write_text(json.dumps(smoke_rows,indent=2))
    del exported
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Final candidate:',CANDIDATE); print('Artifact:',PACKAGES/VERSION); print('Reports:',REPORTS)